This notebook checks the code for importing the disk dictionary and serves as a place to document notes and sources for the values included in the dictionary.

In [7]:
# add the host disk properties 
import pickle
import numpy as np
with open(r'd:\CPD_MPIA\CPD_Emission_Models\disk_arr.pkl', 'rb') as f:
    disk_arr = pickle.load(f)

In [1]:
# Add the rms (mu Jy/beam)
disk_rms = {
    "DM Tau": 26.6,
    "AA Tau": 23.8,
    "LkCa 15": 20.3,
    "HD 34282": 22.6,
    "MWC 758": 31.6,
    "CQ Tau": 25.5,
    "SY Cha": 30.7,
    "PDS 66": 26.0,
    "HD 135344B": 23.4,
    "HD 143006": 25.3,
    "J1604": 23.0,
    "J1615": 19.1,
    "V4046 Sgr": 19.7,
    "J1842": 23.4,
    "J1852": 19.7
}

In [ ]:
# Calculate rout as defined by the radius where brightness drops to 2*rms, and rms is calculated as the median of the MAD between 3*R90 and 5*R90
def calculate_rms_median_mad(profile, r90):
    # profile: array of (radius, brightness)
    # r90: float, R90 value
    # Select radii between 3*R90 and 5*R90
    mask = (profile[:,0] >= 3*r90) & (profile[:,0] <= 5*r90)
    selected_brightness = profile[mask, 1]
    # Calculate MAD
    mad = np.median(np.abs(selected_brightness - np.median(selected_brightness)))
    return mad

def find_rout(profile, rms):
    # Find radius where brightness drops to 2*rms
    for radius, brightness in profile:
        if brightness <= 2*rms:
            return radius
    return None  # If not found

for disk_name, disk_obj in all_disks.items():
    profile = disk_obj.deprojected_profile  # shape (N, 2): radius, brightness
    r90 = getattr(disk_obj, 'R90', None)
    if profile is not None and r90 is not None:
        rms = calculate_rms_median_mad(profile, r90)
        rout = find_rout(profile, 2*rms)
        d['RMS'] = rms
        d['rout'] = rout

In [ ]:


with open(r'd:\CPD_MPIA\Median_SNR\all_disks.pkl', "rb") as f:
    all_disks = pickle.load(f)

all_disk_dicts = {}

for disk_name, disk_obj in all_disks.items():
    d = {}
    d['name'] = disk_name
    d['label'] = disk_name.replace('_', ' ')
    d['distance'] = getattr(disk_obj, 'distance_pc', None)
    d['incl'] = getattr(disk_obj, 'inc', None)
    d['PA'] = getattr(disk_obj, 'PA', None)
    d['dx'], d['dy'] = getattr(disk_obj, 'center', (None, None))
    d['rout'] = getattr(disk_obj, 'disksize', {}).get('R95', None) if hasattr(disk_obj, 'disksize') else None   # in arcsec

    # Add info from disk_arr if available
    if disk_name in disk_arr:
        arr = disk_arr[disk_name]
        d['lstar'] = arr[3]
        d['mstar'] = arr[1] 

    if disk_name in disk_rms:
        d['RMS'] = disk_rms[disk_name] # in microJy/beam
        d['cthres'] = 1 * d['RMS']  # 6-sigma threshold , I actually dont know 


    if hasattr(disk_obj, 'ringgap_info'):
        if hasattr(disk_obj, 'ringgap_info') and "flag" in disk_obj.ringgap_info:
            gaps = disk_obj.ringgap_info['flag'] == 0
            d['rgap'] = (disk_obj.ringgap_info['radius_arcsec'][gaps]/1000).tolist()   # mas
            d['wgap'] = (disk_obj.ringgap_info['width_arcsec'][gaps]/1000).tolist()   # mas
            d['dgap'] = (disk_obj.ringgap_info['gap_depth'][gaps]*100).tolist()  # percentage
    
    else:
        d['rgap'] = []
        d['wgap'] = []
        d['dgap'] = []
        d['rout'] = None


    #  Add Frank fitting parameters and same as DSHARP code
    d['hyp-alpha'] = 1.3
    d['hyp-wsmth'] = 0.1
    d['hyp-Ncoll'] = 300 
    all_disk_dicts[disk_name] = d


    # Add remaining casa properties

    d['cscale'] =  [0, 8, 15, 30, 80]   # pixels  (exoALMA II)
    d['crobust'] = 2.0
    d['ctaper'] = None  # or []
    d['cgain'] = 0.3
    d['ccycleniter'] = 300
    d['cmask'] = f"ellipse[[{arr[6]}, {arr[7]}], [3arcsec, {3 * np.cos(np.radians(d['incl']))}arcsec], {d['PA']}deg]"

    

with open("all_disk_dicts.pkl", "wb") as f:
    pickle.dump(all_disk_dicts, f)

In [ ]:
with open("all_disk_dicts.pkl", "rb") as f:
    all_disk_dicts = pickle.load(f)

# Now you can access the dictionary:
#print(all_disk_dicts["AA_Tau"])

#### check with the Andrews Code   (CASA paramters use exoALMAII)

```
disk = {}

disk['SR4'] =      {'name': 'SR4',   
                    'label': 'SR 4', 
                    'distance': 134.8,
                    'mstar': 0.68,
                    'lstar': 1.17,
                    'incl': 22.0, 
                    'PA': 18.0,
                    'dx': -0.060,  # RA offset in arcsec
                    'dy': -0.509,  # Dec offset in arcsec
                    'rgap': [0.079],  # mas
                    'wgap': [0.010],  # mas
                    'dgap': [30],  # 30 percent flux drop
                    'rout': 0.25, 
                    'maxTb': 50,
                    'hyp-alpha': 1.3,   # frank fitting  - smoothing strength
                    'hyp-wsmth': 0.1, # frank fitting  - smoothing scale
                    'hyp-Ncoll': 300,  # frank fitting  - number of collocation points
                    'cmask': 'circle[[16h25m56.16s, -24.20.48.71], 0.7arcsec]',  # for casa tclean to define imaging region (not full fov)
                    'cscales': [0, 5, 30, 75, 150],    # in very simple words, these are the different "size" of structures that tclean will try to decompose the image into, structures of size ~scale will be modelled with that scale, in units of pixels
		    'gscales': [0, 5],   # for gap specific cleaning
                    'cthresh': '0.05mJy',   
                    'gthresh': '0.034mJy',
                    'crobust': -0.5,
                    'ctaper': ['0.035arcsec', '0.01arcsec', '0deg'],
                    'cgain': 0.3,
                    'ccycleniter': 300,
                    'RMS': 17.8,
                    'peakr': [84.], 
                    'peakaz': [104.]
}
```


    #### Imaging parameters
    1. 'maxTb' =  
    2. 'RMS' = 

    #### Frank fitting
    3. 'hyp-alpha' = 1.3
    4. 'hyp-wsmth' = 0.1
    5. 'hyp-Ncoll' = 300

    #### CASA Imaging Settings:

    'cmask': 'ellipse[...]', # clean mask
    'cscales': [0, 10, 25, 50, 100],  # multiscale clean
    'cthresh': '0.06mJy',   # clean threshold
    'crobust': 0.0,         # robust weighting
    'ctaper': [],           # UV taper
    'cgain': 0.3,           # clean gain
    'ccycleniter': 300,     # clean cycles

    #### Peak Detection
    16. 'peakr'
    17. 'peakaz'   


    #### minimmum needed for CPD search:
    disk['Elias20'] = {
    'name': 'Elias20',
    'incl': 54.0, 
    'PA': 153.2, 
    'dx': -0.052, 
    'dy': -0.490, 
    'rgap': [0.181],
    'wgap': [0.011],
    'rout': 0.48, 
    'hyp-alpha': 1.3,
    'hyp-wsmth': 0.1,
    'hyp-Ncoll': 300,
    'cmask': 'ellipse[[RA, dec], [3, 3cosi], 154deg]',
    'cscales': [0, 10, 25, 50, 100],
    'cthresh': '0.06mJy',    # keep I guess
    'crobust': 2.0,  # robust value
    'ctaper': [],
    'cgain': 0.3,
    'ccycleniter': 300,
}

In [11]:
import pprint
pprint.pprint(all_disk_dicts)

{'AA_Tau': {'PA': 93.77079777,
            'ccycleniter': 150,
            'cgain': 0.3,
            'cmask': 'ellipse[[04 34 55.420, +24 28 53.034], [3arcsec, '
                     '1.5659189121498656arcsec], 93.77079777deg]',
            'crobust': 2.0,
            'cscale': [0, 8, 15, 30, 80],
            'ctaper': None,
            'dgap': [1.0, 44.0, 34.0, 94.0],
            'distance': 135,
            'dx': -0.00545897,
            'dy': 0.00482739,
            'hyp-Ncoll': 300,
            'hyp-alpha': 1.3,
            'hyp-wsmth': 0.1,
            'incl': 58.53531224,
            'label': 'AA Tau',
            'lstar': 1.1,
            'mstar': 0.79,
            'name': 'AA_Tau',
            'rgap': [8.2e-05, 0.00047799999999999996, 0.000593, 0.000782],
            'rout': np.float64(1.177),
            'wgap': [0.00020899999999999998,
                     6.1e-05,
                     7.6e-05,
                     3.5999999999999994e-05]},
 'CQ_Tau': {'PA': 53.87180444,
    